In [1]:
import os
import glob
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
IMAGE_DIR = "../datasets/sampled_sneakers_1024x1024"

In [3]:
exts = ["*.jpg", "*.jpeg", "*.png", "*.webp"]

image_paths = []

for ext in exts:
    image_paths.extend(glob.glob(os.path.join(IMAGE_DIR, ext)))

print("Total images:", len(image_paths))

Total images: 2642


In [4]:
brands = [
    "nike",
    "puma",
    "vans",
    "newbalance",
    "underarmour",
    "reebok"
]

def get_brand(path):
    name = os.path.basename(path).lower()
    
    for brand in brands:
        if brand in name:
            return brand
            
    return None

In [5]:
df = pd.DataFrame({
    "image_path": image_paths
})

df["image_name"] = df["image_path"].apply(os.path.basename)
df["brand"] = df["image_path"].apply(get_brand)

df.head()

,image_path,image_name,brand
0,../datasets/sampled_sneakers_1024x1024\normali...,normalized_nike_0.jpg,nike
1,../datasets/sampled_sneakers_1024x1024\normali...,normalized_nike_1.jpg,nike
2,../datasets/sampled_sneakers_1024x1024\normali...,normalized_nike_10.jpg,nike
3,../datasets/sampled_sneakers_1024x1024\normali...,normalized_nike_11.jpg,nike
4,../datasets/sampled_sneakers_1024x1024\normali...,normalized_nike_112.jpg,nike


In [6]:
print("Images without brand:", df["brand"].isna().sum())

if df["brand"].isna().sum() > 0:
    print(df[df["brand"].isna()]["image_name"].tolist())

Images without brand: 0


In [7]:
print(df["brand"].value_counts())

brand
vans           919
puma           547
nike           379
reebok         341
underarmour    230
newbalance     226
Name: count, dtype: int64


In [8]:
train_df, test_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["brand"],
    random_state=42
)

In [9]:
print("Train:", len(train_df))
print("Test :", len(test_df))

Train: 2245
Test : 397


In [10]:
train_df, val_df = train_test_split(
    train_df,
    test_size=0.17647,
    stratify=train_df["brand"],
    random_state=42
)

In [11]:
print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

Train: 1848
Val  : 397
Test : 397
Total: 2642


In [12]:
print("TRAIN")
print(train_df["brand"].value_counts())
print()

print("VALIDATION")
print(val_df["brand"].value_counts())
print()

print("TEST")
print(test_df["brand"].value_counts())

TRAIN
brand
vans           643
puma           383
nike           265
reebok         239
underarmour    160
newbalance     158
Name: count, dtype: int64

VALIDATION
brand
vans           138
puma            82
nike            57
reebok          51
underarmour     35
newbalance      34
Name: count, dtype: int64

TEST
brand
vans           138
puma            82
nike            57
reebok          51
underarmour     35
newbalance      34
Name: count, dtype: int64


In [13]:
for name, d in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}")
    print(
        d["brand"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )


Train
brand
vans           34.79
puma           20.73
nike           14.34
reebok         12.93
underarmour     8.66
newbalance      8.55
Name: proportion, dtype: float64

Validation
brand
vans           34.76
puma           20.65
nike           14.36
reebok         12.85
underarmour     8.82
newbalance      8.56
Name: proportion, dtype: float64

Test
brand
vans           34.76
puma           20.65
nike           14.36
reebok         12.85
underarmour     8.82
newbalance      8.56
Name: proportion, dtype: float64


In [14]:
train_imgs = set(train_df["image_name"])
val_imgs = set(val_df["image_name"])
test_imgs = set(test_df["image_name"])

print("Train ∩ Val :", len(train_imgs & val_imgs))
print("Train ∩ Test:", len(train_imgs & test_imgs))
print("Val ∩ Test  :", len(val_imgs & test_imgs))

Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0


In [15]:
assert len(train_imgs & val_imgs) == 0
assert len(train_imgs & test_imgs) == 0
assert len(val_imgs & test_imgs) == 0

print("No image-level leakage!")

No image-level leakage!


In [16]:
SPLIT_DIR = "../datasets/splits"

os.makedirs(SPLIT_DIR, exist_ok=True)

train_df.to_csv(
    os.path.join(SPLIT_DIR, "train_images.csv"),
    index=False
)

val_df.to_csv(
    os.path.join(SPLIT_DIR, "val_images.csv"),
    index=False
)

test_df.to_csv(
    os.path.join(SPLIT_DIR, "test_images.csv"),
    index=False
)

print("Splits saved!")

Splits saved!
